# 🔍⚡ KVCacheScope: Live vLLM KV Cache Profiler & Inspector
### Visualizing Logical Block Fragmentation, Prefix Caching Radix Trees & Hostage Leaks on Real GPUs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vllm-project/kvcachescope)

Standard GPU profilers (`nsys`, `pmap`, `Heaptrack`) only observe raw, physical GPU memory allocations, entirely missing the **logical block tables** maintained inside an inference engine's internal memory manager.

This notebook connects **KVCacheScope** directly to a live running **vLLM engine** to observe:
1. **Real GPU Block Tables**: Tracking exact physical VRAM block IDs as tokens decode.
2. **Prefix Caching Radix Trees**: Real-time reference counting and deduplication across prompts.
3. **Internal Slack Waste**: Token-level slack in tail blocks.
4. **Hostage Zombie Leaks**: Ungraceful client disconnects leaving memory locked in the pool.

## 📦 Step 1: Install Dependencies
Install `vllm`, `nest_asyncio`, `fastapi`, `uvicorn`, and `websockets`.

In [ ]:
# Install vLLM and server packages
!pip install -q vllm nest_asyncio fastapi uvicorn websockets pydantic

## 🛠️ Step 2: Download KVCacheScope
Fetch the KVCacheScope backend and pre-built React dashboard.

In [ ]:
!git clone https://github.com/vllm-project/kvcachescope.git /content/KVCacheScope || true
%cd /content/KVCacheScope

## 🚀 Step 3: Apply `nest_asyncio` & Start KVCacheScope Server
Colab runs inside an active IPython asyncio event loop. We apply `nest_asyncio` to allow FastAPI and WebSockets to run concurrently.

In [ ]:
import nest_asyncio
nest_asyncio.apply()

import threading
import uvicorn
from backend.server import app

def start_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

# Start web server in background thread
server_thread = threading.Thread(target=start_server, daemon=True)
server_thread.start()
print("✅ KVCacheScope Server running on port 8000!")

## 🌐 Step 4: Expose Live Interactive Dashboard
Using Google Colab's native `serve_kernel_port_as_window` to open the full interactive UI in a separate window or inline frame.

In [ ]:
from google.colab import output
output.serve_kernel_port_as_window(8000)
print("🔗 Click the popup link above or inspect the embedded iframe below:")

# Also render inline frame
from IPython.display import IFrame, display
# display(IFrame('http://localhost:8000', width=1200, height=700))

## 🤖 Step 5: Initialize Live vLLM Engine & Attach Telemetry Hook
Load an open-source model (e.g. `facebook/opt-125m` or `TinyLlama/TinyLlama-1.1B-Chat-v1.0`) with `enable_prefix_caching=True` and attach the `KVCacheScopeVLLMHook`.

In [ ]:
import time
from vllm import LLMEngine, EngineArgs, SamplingParams
from backend.vllm_hook import attach_vllm_hook

print("[*] Initializing real vLLM engine on GPU...")
engine_args = EngineArgs(
    model="facebook/opt-125m",
    enable_prefix_caching=True,
    max_num_seqs=16,
    gpu_memory_utilization=0.6
)
llm_engine = LLMEngine.from_engine_args(engine_args)

# Attach live telemetry hook
hook = attach_vllm_hook(llm_engine, port=8000)
print("✅ Hook attached to live BlockSpaceManager! Streaming physical GPU block tables at 10Hz.")

## 🔥 Step 6: Run Continuous Batching & Observe Live GPU Memory Pools
Fire multi-turn prompts sharing common prefixes, simulate dropped sessions, and watch real VRAM blocks update live on your KVCacheScope dashboard.

In [ ]:
import random

SYSTEM_PROMPT = "You are an expert systems engineer specializing in LLM inference acceleration, CUDA kernels, and memory managers. "
QUERIES = [
    "Explain how PagedAttention block tables map virtual tokens to physical GPU memory.",
    "Draft an RFC for disaggregated prefill and decode KV cache transfer over RDMA.",
    "Analyze the memory consumption profile of vLLM worker nodes under 8k context length.",
    "Compare prefix caching radix trees with hash-based exact block lookup tables.",
    "Write a CUDA kernel for block-sparse attention gathering on Ampere architectures."
]

sampling_params = SamplingParams(temperature=0.7, top_p=0.9, max_tokens=48)

print("[*] Generating live batch traffic to vLLM (Observe KVCacheScope window!)...")
for i in range(25):
    # Submit request
    query = random.choice(QUERIES)
    prompt = f"{SYSTEM_PROMPT} Query {i}: {query}"
    req_id = f"colab_req_{i}"
    
    llm_engine.add_request(req_id, prompt, sampling_params)
    print(f"[+] Submitted request {req_id} (Shared prefix prompt)")
    
    # Step inference
    for _ in range(4):
        if llm_engine.has_unfinished_requests():
            step_outputs = llm_engine.step()
            for out in step_outputs:
                if out.finished:
                    print(f"[✓] Finished {out.request_id} -> Block table freed!")
        time.sleep(0.05)

print("\n🎉 Benchmark completed! Open the KVCacheScope window to inspect the final memory state.")